# **02_TRANS — Transformación y Calidad de Datos (Capa Silver)**

## Descripción
Esta notebook implementa la capa Silver de la arquitectura Medallón.
A partir del dato crudo ingestado en Bronze, se ejecutan procesos de 
diagnóstico, limpieza y estandarización para garantizar que los datos 
cumplan los criterios de calidad antes de pasar al modelado dimensional.

## Criterios de la Capa Silver
| Criterio | Descripción |
|---|---|
| 0 nulos sin tratar | Campos críticos limpiados o eliminados con justificación |
| 0 duplicados en PK | `transaction_id` validado como clave primaria única |
| Tipos de dato correctos | Columnas casteadas a sus tipos correspondientes |
| Idempotencia | Todos los objetos incluyen `DROP IF EXISTS` |

## Tablas Generadas
| Tabla | Descripción |
|---|---|
| `silver_sucio` | Dataset con problemas de calidad simulados para demostrar limpieza |
| `silver` | Dataset limpio y listo para modelado dimensional |

## Flujo de la Notebook
1. Carga de `ventas_raw` desde Bronze
2. Simulación de problemas de calidad (`silver_sucio`)
3. Diagnóstico de nulos y duplicados
4. Limpieza y creación de tabla `silver`
5. Validación del resultado: conteos antes y después


## **Carga y Tipado — dim_categoria**
Se carga la dimensión desde Silver y se garantizan los tipos de dato 
correctos antes de escribirla en Gold.

| Columna | Tipo | Descripción |
|---|---|---|
| `id_categoria` | `IntegerType` | Clave surrogada numérica |
| `product_category` | `StringType` | Nombre de la categoría de producto |

In [1]:
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.functions import col

dim_categoria = spark.read.table("silver.dbo.dim_categoria")

dim_categoria = dim_categoria \
    .withColumn("id_categoria",     col("id_categoria").cast(IntegerType())) \
    .withColumn("product_category", col("product_category").cast(StringType()))

display(dim_categoria)

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8263657-5dae-462f-9394-5c00fa40e1be)

## **Carga y Tipado — dim_fecha**
Se carga la dimensión desde Silver y se garantizan los tipos de dato 
correctos para cada atributo temporal antes de escribirla en Gold.

| Columna | Tipo | Descripción |
|---|---|---|
| `id_fecha` | `IntegerType` | Clave surrogada numérica |
| `fecha` | `DateType` | Fecha completa en formato `yyyy-MM-dd` |
| `anio` | `ShortType` | Año extraído de la fecha |
| `mes` | `ShortType` | Mes extraído de la fecha (1–12) |
| `dia` | `ShortType` | Día extraído de la fecha (1–31) |

> 💡 `ShortType` se usa para `anio`, `mes` y `dia` por ser valores 
> numéricos pequeños, optimizando el almacenamiento en Gold.

In [2]:
from pyspark.sql.types import IntegerType, ShortType, DateType, StringType
from pyspark.sql.functions import col, to_date, month, when

dim_fecha = spark.read.table("silver.dbo.dim_fecha")

dim_fecha = dim_fecha \
    .withColumn("id_fecha",  col("id_fecha").cast(IntegerType())) \
    .withColumn("fecha",     to_date(col("fecha"), "yyyy-MM-dd")) \
    .withColumn("anio",      col("anio").cast(ShortType())) \
    .withColumn("mes",       col("mes").cast(ShortType())) \
    .withColumn("dia",       col("dia").cast(ShortType())) \
    .withColumn("nombre_mes",
        when(col("mes") == 1,  "Enero")
        .when(col("mes") == 2,  "Febrero")
        .when(col("mes") == 3,  "Marzo")
        .when(col("mes") == 4,  "Abril")
        .when(col("mes") == 5,  "Mayo")
        .when(col("mes") == 6,  "Junio")
        .when(col("mes") == 7,  "Julio")
        .when(col("mes") == 8,  "Agosto")
        .when(col("mes") == 9,  "Septiembre")
        .when(col("mes") == 10, "Octubre")
        .when(col("mes") == 11, "Noviembre")
        .when(col("mes") == 12, "Diciembre")
    )

display(dim_fecha)

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c58e1ee0-cef3-499c-979d-6ea1a8f3a2f2)

## **Carga y Tipado — dim_genero**
Se carga la dimensión desde Silver aplicando normalización adicional 
antes de escribirla en Gold.

| Columna | Tipo | Transformación aplicada |
|---|---|---|
| `id_genero` | `IntegerType` | Regenerado con `ROW_NUMBER()` tras deduplicación |
| `nombre_genero` | `StringType` | Normalizado con `initcap` → `Female`, `Male` |

**Pasos aplicados:**
1. `initcap` — estandariza capitalización (`female` → `Female`, `MALE` → `Male`)
2. `dropDuplicates` — elimina géneros duplicados que pudieron quedar tras la limpieza
3. `row_number()` — regenera la clave surrogada limpia y ordenada alfabéticamente

> 💡 Esta normalización es necesaria porque `silver_sucio` introdujo 
> variaciones de mayúsculas en `gender` que pudieron sobrevivir 
> parcialmente a la limpieza.

In [3]:
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.functions import col, initcap, monotonically_increasing_id, row_number
from pyspark.sql.window import Window

dim_genero = spark.read.table("silver.dbo.dim_genero")

dim_genero = dim_genero \
    .withColumn("nombre_genero", initcap(col("nombre_genero").cast(StringType()))) \
    .dropDuplicates(["nombre_genero"]) \
    .withColumn("id_genero", row_number().over(Window.orderBy("nombre_genero")).cast(IntegerType()))

display(dim_genero)
print(dim_genero.printSchema())

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e468b047-db28-439b-a655-ef21a6e91f27)

root
 |-- id_genero: integer (nullable = false)
 |-- nombre_genero: string (nullable = true)

None


## **Carga y Tipado — hechos_ventas**
Se carga la tabla de hechos desde Silver aplicando casteo explícito 
a todos los campos para garantizar consistencia de tipos en Gold.

| Columna | Tipo | Descripción |
|---|---|---|
| `id_genero` | `IntegerType` | FK hacia `dim_genero` — normalizada a 1 o 2 |
| `transaction_id` | `StringType` | Identificador único de la transacción |
| `customer_id` | `StringType` | Identificador del cliente |
| `id_categoria` | `IntegerType` | FK hacia `dim_categoria` |
| `id_fecha` | `IntegerType` | FK hacia `dim_fecha` |
| `quantity` | `IntegerType` | Unidades vendidas |
| `price_per_unit` | `DoubleType` | Precio unitario del producto |
| `total_amount` | `DoubleType` | Valor total de la transacción |
| `age` | `IntegerType` | Edad del cliente |

> 💡 `CASE WHEN id_genero = 1 THEN 1 ELSE 2 END` garantiza que la FK 
> solo contenga los valores válidos `1` y `2`, evitando registros 
> huérfanos al hacer JOIN con `dim_genero` en Gold.

In [4]:
from pyspark.sql.types import IntegerType, StringType, DoubleType

hechos_ventas = spark.sql("""
    SELECT
        CASE WHEN id_genero = 1 THEN 1 ELSE 2 END AS id_genero,
        CAST(transaction_id  AS STRING)  AS transaction_id,
        CAST(customer_id     AS STRING)  AS customer_id,
        CAST(id_categoria    AS INT)     AS id_categoria,
        CAST(id_fecha        AS INT)     AS id_fecha,
        CAST(quantity        AS INT)     AS quantity,
        CAST(price_per_unit  AS DOUBLE)  AS price_per_unit,
        CAST(total_amount    AS DOUBLE)  AS total_amount,
        CAST(age  AS INT)  AS age
        

    FROM silver.dbo.Hechos_Ventas
""")

display(hechos_ventas)

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50b3212b-ac67-4857-ae02-8781b0bbe4f6)

## **Verificación del Schema — hechos_ventas**
Imprime la estructura del DataFrame para confirmar que todos los tipos 
de dato fueron aplicados correctamente antes de persistir en Gold.

In [5]:
print(hechos_ventas)

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 7, Finished, Available, Finished, False)

DataFrame[id_genero: int, transaction_id: string, customer_id: string, id_categoria: int, id_fecha: int, quantity: int, price_per_unit: double, total_amount: double, age: int]


## **Persistencia en Gold — dim_categoria**
Se escribe la dimensión en la capa Gold en formato Delta.

> ⚠️ El modo `overwrite` garantiza idempotencia — el notebook puede 
> re-ejecutarse sin generar duplicados.

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


dim_categoria.write.mode("overwrite")\
        .format("delta")\
        .saveAsTable("dim_categoria")

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 8, Finished, Available, Finished, False)

## **Persistencia en Gold — dim_fecha**
Se escribe la dimensión calendario en la capa Gold en formato Delta.

> ⚠️ El modo `overwrite` garantiza idempotencia — el notebook puede 
> re-ejecutarse sin generar duplicados.

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

dim_fecha.write.mode("overwrite")\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .saveAsTable("dim_fecha")

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 9, Finished, Available, Finished, False)

## **Persistencia en Gold — dim_genero**
Se escribe la dimensión de géneros en la capa Gold en formato Delta.

> ⚠️ El modo `overwrite` garantiza idempotencia — el notebook puede 
> re-ejecutarse sin generar duplicados.

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


dim_genero.write.mode("overwrite")\
        .format("delta")\
        .saveAsTable("dim_genero")

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 10, Finished, Available, Finished, False)

## **Persistencia en Gold — hechos_ventas**
Se escribe la tabla de hechos en la capa Gold en formato Delta.
Con esta escritura el modelo estrella queda completo y disponible 
para consumo analítico, modelo semántico y capa de visualización.

> ⚠️ El modo `overwrite` garantiza idempotencia — el notebook puede 
> re-ejecutarse sin generar duplicados.

In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


hechos_ventas.write.mode("overwrite")\
        .format("delta")\
        .saveAsTable("Hechos_Ventas")

StatementMeta(, 0db8755b-e9aa-4b0b-aaed-a5b45a9cd711, 11, Finished, Available, Finished, False)